# Phase 3c — Approach 3: ResNet-50 Fine-Tuning

**CSC3109 Machine Learning | Group 30**

## What Is Fine-Tuning?

Fine-tuning goes one step further than feature extraction (Approach 2). Instead of keeping all ResNet-50 layers frozen, we **unfreeze the later layers** and allow them to be adjusted for our specific domain (aerial imagery).

The intuition: the early layers of ResNet-50 detect low-level features (edges, colour gradients) that are universal across all image types. The later layers detect high-level patterns (specific textures, object parts) that are more domain-specific — so we let those adapt.

```
ResNet-50
+-- layer1 (frozen)            detects edges, colour -- universal
+-- layer2 (frozen)            detects textures -- mostly universal
+-- layer3 (frozen)            detects shapes -- mostly universal
+-- layer4 (UNFROZEN, lr=1e-5) high-level features -- domain-specific, we adapt these
+-- fc     (UNFROZEN, lr=1e-4) classification head
```

**Why different learning rates?** `layer4` already has good ImageNet features — we only nudge them slightly (lr=1e-5). The `fc` layer is new and needs to learn more aggressively (lr=1e-4).

In [ ]:
import os
import sys
from pathlib import Path

# AUTO-DETECT LOCAL vs COLAB
if os.path.exists('/content'):
    REPO_ROOT = Path('/content/csc3109-g30')
    print('Detected: Colab environment')
else:
    REPO_ROOT = Path('..').resolve()
    print('Detected: Local environment')

sys.path.insert(0, str(REPO_ROOT / 'src'))

# Verify paths
assert (REPO_ROOT / 'src' / 'dataset.py').exists(), f"dataset.py not found"
assert (REPO_ROOT / 'data' / 'train').exists(), f"train data not found"

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from dataset import get_data_loaders
from train_utils import train_model, plot_training_curves, plot_confusion_matrix, \
                        get_all_predictions, print_classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPORT_DIR = REPO_ROOT / 'report'
REPORT_DIR.mkdir(exist_ok=True)
MODELS_DIR = REPO_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {DEVICE}')
print('All imports OK')

---
## Step 1 — Load Data

In [ ]:
train_loader, val_loader, cls2idx, idx2cls = get_data_loaders(
    data_dir=REPO_ROOT / 'data',
    batch_size=32,
    num_workers=0
)

NUM_CLASSES = len(cls2idx)
print(f'\nClass mapping: {cls2idx}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'\nExpected: 600 train / 100 val per class')
print('If counts above are wrong, re-run Phase 1 (01_eda.ipynb) to recreate the split.')

---
## Step 2 — Load ResNet-50 and Set Up for Fine-Tuning

We have two options:
- **Option A:** Start from ImageNet weights (cold start)
- **Option B:** Start from our Approach 2 checkpoint (warm start — recommended)

Option B is better because Approach 2 already trained the FC head on our data. Fine-tuning from there means the head is already well-initialised.

In [ ]:
# Load ResNet-50 architecture
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Option B: warm start from Approach 2 checkpoint (run 03b first)
checkpoint_path = MODELS_DIR / 'resnet50_extraction_best.pth'
if checkpoint_path.exists():
    model.load_state_dict(torch.load(checkpoint_path, map_location='cpu'))
    print('Loaded Approach 2 checkpoint — warm start')
else:
    print('Approach 2 checkpoint not found — cold start from ImageNet weights')

# Start with everything frozen
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 and the FC head
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
print('(layer4 + FC unfrozen)')

---
## Step 3 — Configure Training with Differential Learning Rates

In [ ]:
NUM_EPOCHS = 20

# Differential learning rates: lower for layer4, higher for FC
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(),     'lr': 1e-4},
])

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-7
)

print('Differential learning rates:')
print('  layer4 (adapting high-level features):  lr = 1e-5')
print('  FC head (learning classification):       lr = 1e-4')
print('Scheduler: Cosine Annealing (smooth decay to near-zero)')

---
## Step 4 — Train

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_name='resnet50_finetune',
    device=DEVICE
)

---
## Step 5 — Plot and Evaluate

In [ ]:
plot_training_curves(
    history,
    title='Approach 3 — ResNet-50 Fine-Tuning',
    save_path=REPORT_DIR / 'approach3_resnet50_finetune_curves.png'
)

In [ ]:
all_labels, all_preds = get_all_predictions(model, val_loader, DEVICE)
print('=== Approach 3: ResNet-50 Fine-Tuning — Validation Results ===')
print_classification_report(all_labels, all_preds)
print(f'Overall Accuracy: {(all_labels == all_preds).mean():.4f}')

In [ ]:
plot_confusion_matrix(
    all_labels, all_preds,
    title='Approach 3 — ResNet-50 Fine-Tuning',
    save_path=REPORT_DIR / 'approach3_resnet50_finetune_confusion.png'
)

---
## Observations for Report

**Expected outcome:** Fine-tuning should outperform feature extraction because layer4 can now adapt to aerial-imagery-specific patterns.

**Key comparison:** Compare Approach 2 vs Approach 3 accuracy — the difference quantifies how much domain adaptation helps.

**Strengths:** Domain-adapted high-level features, good generalisation, moderate training time  
**Weaknesses:** Slightly higher risk of overfitting than feature extraction, requires careful LR tuning

Best model saved to `models/resnet50_finetune_best.pth`

**Next:** `03d_efficientnet.ipynb`